# Notebook 22 — SHAP Attribution
### Heterogeneous Treatment Effects in Mortgage Lending

**Author:** Rajveer Singh Pall  
**Institution:** Gyan Ganga Institute of Technology and Sciences

---

Identifies which covariates drive heterogeneity in the CATE estimates from NB21.
Trains a gradient boosting model to predict CATE from X, then applies
SHAP TreeExplainer to decompose each prediction into feature contributions.

**INPUT:** `data/cate_estimates.parquet`, `data/features_panel.parquet`, `data/feature_sets.json`

**OUTPUTS:** `nb22_shap_beeswarm.png`, `nb22_shap_bar.png`, `nb22_shap_importance.csv`

**RUNTIME:** 15-25 minutes

In [ ]:
# CELL 1 - IMPORTS
import pandas as pd
import numpy as np
import polars as pl
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import json, gc, warnings
from pathlib import Path
from scipy.stats import binned_statistic

warnings.filterwarnings('ignore')

BASE_DIR    = Path('D:/CATE-HMDA-Heterogeneous-Effects')
DATA_DIR    = BASE_DIR / 'data'
TABLES_DIR  = BASE_DIR / 'outputs' / 'tables'
FIGURES_DIR = BASE_DIR / 'outputs' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.family': 'serif', 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

with open(DATA_DIR / 'feature_sets.json') as f:
    feature_sets = json.load(f)
X_FULL = feature_sets['X_FULL']

# Human-readable feature names for plots
FNAME = {
    'aus_automated': 'Automated underwriting (AUS)',
    'aus_exempt': 'Manual/exempt underwriting',
    'ltv': 'Loan-to-value ratio',
    'income': 'Applicant income',
    'log_income': 'Log income',
    'dti_midpoint': 'Debt-to-income ratio',
    'dti_high': 'High DTI (>=43%)',
    'above_pmi': 'Above PMI threshold (LTV>80%)',
    'near_pmi_threshold': 'Near PMI threshold',
    'post_tightening': 'Post-2022 tightening',
    'purpose_purchase': 'Purchase loan',
    'purpose_refi': 'Refinance loan',
    'lender_small': 'Small lender',
    'lender_mid': 'Mid-size lender',
    'lender_large': 'Large lender',
    'applicant_age_mid': 'Applicant age',
    'year': 'Application year',
    'log_loan_amount': 'Log loan amount',
    'log_property_val': 'Log property value',
    'loan_amount': 'Loan amount',
    'property_value': 'Property value',
    'type_fha': 'FHA loan type',
    'type_va': 'VA loan type',
    'type_conventional': 'Conventional loan',
    'type_usda': 'USDA loan type',
    'occ_primary': 'Primary residence',
    'occ_second': 'Second home',
    'occ_investment': 'Investment property',
    'dti_missing': 'DTI missing flag',
    'purpose_homeimprv': 'Home improvement loan',
    'high_minority_tract': 'High minority tract',
    'low_minority_tract': 'Low minority tract',
    'tract_minority_pct': 'Tract minority %',
}

print(f'SHAP version : {shap.__version__}')
assert (DATA_DIR / 'cate_estimates.parquet').exists(), 'Run NB21 first'
print('Input files  : OK')

In [ ]:
# CELL 2 - LOAD CATE ESTIMATES AND FEATURES
print('='*70)
print('LOADING DATA')
print('='*70)

# Load CATE estimates from NB21
cate_df = pl.read_parquet(str(DATA_DIR / 'cate_estimates.parquet')).to_pandas()
print(f'CATE estimates: {len(cate_df):,} rows')
print(f'CATE mean     : {cate_df["cate_pp"].mean():.2f} pp')
print(f'CATE std      : {cate_df["cate_pp"].std():.2f} pp')
print(f'CATE cols     : {list(cate_df.columns)}')

# Use cate_df directly for SHAP — it has key features already
# Identify which X_FULL features are in cate_df
X_use = [f for f in X_FULL if f in cate_df.columns]
print(f'\nFeatures available in cate_df: {X_use}')

# If too few features in cate_df, augment with features_panel sample
if len(X_use) < 8:
    print('Augmenting with features_panel...')
    lf = pl.scan_parquet(str(DATA_DIR / 'features_panel.parquet'))
    df_b = lf.filter(pl.col('black')==1).collect().sample(n=60_000, seed=789)
    df_w = lf.filter(pl.col('black')==0).collect().sample(n=240_000, seed=789)
    df_aug = pl.concat([df_b, df_w]).to_pandas()
    del df_b, df_w, lf
    gc.collect()
    X_use = [f for f in X_FULL if f in df_aug.columns]
    df_shap = df_aug.sample(n=min(200_000, len(df_aug)), random_state=789).reset_index(drop=True)
    del df_aug
    gc.collect()
    # Get CATE for these rows by sampling cate_df
    cate_target = cate_df['cate_pp'].sample(n=len(df_shap), random_state=789).values
else:
    df_shap = cate_df.sample(n=min(200_000, len(cate_df)), random_state=789).reset_index(drop=True)
    cate_target = df_shap['cate_pp'].values

gc.collect()
print(f'\nSHAP dataset  : {len(df_shap):,} rows x {len(X_use)} features')
print(f'Features      : {X_use}')
print(f'RAM           : {df_shap.memory_usage(deep=True).sum()/1e6:.0f} MB')

In [ ]:
# CELL 3 - TRAIN CATE PREDICTOR
# Train LightGBM to predict CATE from features.
# SHAP decomposes this model's predictions into feature contributions.
print('='*70)
print('TRAINING CATE PREDICTOR FOR SHAP')
print('='*70)

X_mat = df_shap[X_use].values.astype(np.float32)
y_mat = cate_target.astype(np.float32)

cate_predictor = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, verbose=-1
)
cate_predictor.fit(X_mat, y_mat)

from sklearn.metrics import r2_score
preds = cate_predictor.predict(X_mat)
r2    = r2_score(y_mat, preds)
print(f'R² (train)  : {r2:.3f}')
print(f'Pred mean   : {preds.mean():.2f} pp')
print(f'Target mean : {y_mat.mean():.2f} pp')

if r2 < 0.3:
    print('NOTE: R2 below 0.3 — CATE surface is noisy (expected for causal forests)')
    print('SHAP will still correctly identify direction of effects')
print('CATE predictor trained')

In [ ]:
# CELL 4 - COMPUTE SHAP VALUES
print('='*70)
print('COMPUTING SHAP VALUES')
print('='*70)

explainer   = shap.TreeExplainer(cate_predictor)
shap_values = explainer.shap_values(X_mat)

print(f'SHAP matrix shape: {shap_values.shape}')

# Feature importance
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'feature':       X_use,
    'display_name':  [FNAME.get(f, f) for f in X_use],
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('\nTop 10 features driving CATE heterogeneity:')
for i, row in importance_df.head(10).iterrows():
    print(f'  {i+1:2d}. {row["display_name"]:<40} {row["mean_abs_shap"]:.4f}')

importance_df.to_csv(TABLES_DIR / 'nb22_shap_importance.csv', index=False)
print('\nSaved: nb22_shap_importance.csv')

In [ ]:
# CELL 5 - SHAP BEESWARM PLOT
print('Generating SHAP beeswarm plot...')

display_names = [FNAME.get(f, f) for f in X_use]

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_mat,
    feature_names=display_names,
    plot_type='dot',
    max_display=15,
    show=False,
)
plt.title(
    'Figure NB22-1: SHAP Attribution — Drivers of CATE Heterogeneity\n'
    'Features ranked by mean |SHAP| · Left = larger penalty · Color = feature value',
    fontsize=10, pad=12
)
plt.xlabel('SHAP value (impact on racial approval penalty, pp)')
plt.tight_layout()
out = FIGURES_DIR / 'nb22_shap_beeswarm.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')

In [ ]:
# CELL 6 - SHAP BAR CHART (CLEAN VERSION FOR PAPER)
print('Generating SHAP bar chart...')

top_n    = min(12, len(importance_df))
plot_imp = importance_df.head(top_n).sort_values('mean_abs_shap').copy()

# Color by channel
discretion = {'aus_exempt', 'aus_automated', 'lender_small', 'lender_large', 'lender_mid'}
financial  = {'ltv', 'income', 'log_income', 'dti_midpoint', 'dti_high',
              'loan_amount', 'log_loan_amount', 'property_value', 'log_property_val'}
colors = []
for f in plot_imp['feature']:
    if f in discretion:
        colors.append('#E53935')
    elif f in financial:
        colors.append('#1565C0')
    else:
        colors.append('#37474F')

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(range(top_n), plot_imp['mean_abs_shap'], color=colors, alpha=0.85)
ax.set_yticks(range(top_n))
ax.set_yticklabels(plot_imp['display_name'], fontsize=9)
ax.set_xlabel('Mean |SHAP| — contribution to CATE heterogeneity (pp)', fontsize=10)
ax.set_title('Figure NB22-2: Feature Importance for CATE Heterogeneity', fontsize=11)

for i, (_, row) in enumerate(plot_imp.iterrows()):
    ax.text(row['mean_abs_shap'] + 0.002, i,
            f"{row['mean_abs_shap']:.3f}", va='center', fontsize=8)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#E53935', alpha=0.85, label='Institutional discretion channel'),
    Patch(color='#1565C0', alpha=0.85, label='Financial characteristics'),
    Patch(color='#37474F', alpha=0.85, label='Other'),
], fontsize=9)

plt.tight_layout()
out = FIGURES_DIR / 'nb22_shap_bar.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')

In [ ]:
# CELL 7 - DEPENDENCE PLOTS FOR TOP 3 FEATURES
print('Generating dependence plots...')

top3 = importance_df['feature'].values[:3]
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for idx, feat in enumerate(top3):
    if feat not in X_use:
        continue
    feat_idx  = X_use.index(feat)
    ax        = axes[idx]
    feat_vals = X_mat[:, feat_idx]
    shap_feat = shap_values[:, feat_idx]

    ax.scatter(feat_vals, shap_feat, alpha=0.15, s=2, color='#1565C0', rasterized=True)

    # Trend line
    try:
        bin_means, bin_edges, _ = binned_statistic(feat_vals, shap_feat, statistic='mean', bins=25)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        valid = ~np.isnan(bin_means)
        ax.plot(bin_centers[valid], bin_means[valid], 'r-', linewidth=2.5, label='Mean trend')
    except Exception:
        pass

    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_xlabel(FNAME.get(feat, feat), fontsize=9)
    ax.set_ylabel('SHAP value (pp)', fontsize=9)
    ax.set_title(f'Effect of\n{FNAME.get(feat, feat)}', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle(
    'Figure NB22-3: SHAP Dependence Plots — Top 3 CATE Drivers\n'
    'Negative SHAP = larger racial approval penalty',
    fontsize=10
)
plt.tight_layout()
out = FIGURES_DIR / 'nb22_shap_dependence.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')

In [ ]:
# CELL 8 - VERIFICATION
print('='*70)
print('VERIFICATION')
print('='*70)

assert shap_values.shape[0] == len(df_shap)
assert shap_values.shape[1] == len(X_use)
print(f'1. SHAP matrix shape: {shap_values.shape} OK')

for f in ['nb22_shap_beeswarm.png', 'nb22_shap_bar.png', 'nb22_shap_dependence.png']:
    assert (FIGURES_DIR / f).exists()
assert (TABLES_DIR / 'nb22_shap_importance.csv').exists()
print(f'2. All output files present OK')

top_feature = importance_df.iloc[0]['feature']
top_name    = importance_df.iloc[0]['display_name']
print(f'3. Top SHAP driver: {top_name}')

print(f'\nFull importance ranking:')
for _, row in importance_df.iterrows():
    print(f'  {row.name+1:2d}. {row["display_name"]:<40} {row["mean_abs_shap"]:.4f}')

print('\n' + '='*70)
print('ALL CHECKS PASSED')
print(f'Top driver: {top_name}')
print(f'NB22 complete -> proceed to NB23_disparity_map.ipynb')
print('='*70)